# Daily Divvy Ridership Forecasting Model

#### Build and Score Baseline

In [13]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# Load the 2025 daily dataset
df = pd.read_csv("daily_ridership_weather_2025.csv")
df.head()

,date,total_rides,member_rides,casual_rides,electric_rides,classic_rides,temperature_2m_max (°F),temperature_2m_min (°F),precipitation_sum (inch),snowfall_sum (inch),rain_sum (inch),temperature_2m_mean (°F),wind_speed_10m_max (mp/h)
0,2025-01-01,3562,2321,1241,2280,1282,33.4,25.5,0.004,0.028,0.000,29.7,13.8
1,2025-01-02,4749,3807,942,2933,1816,31.5,24.5,0.008,0.055,0.000,27.7,10.8
2,2025-01-03,4200,3436,764,2690,1510,26.0,17.9,0.000,0.000,0.000,22.8,12.9
3,2025-01-04,3198,2479,719,1966,1232,22.9,12.2,0.000,0.000,0.000,17.7,10.8
4,2025-01-05,2933,2370,563,1888,1045,27.9,15.7,0.024,0.138,0.004,20.5,13.6


In [14]:
df["baseline_pred"] = df["total_rides"].shift(7)

# 3. Split chronologically (first 70% past days, last 30% future test days)
split_index = int(len(df) * 0.7)
test_df = df.iloc[split_index:]

baseline_mae = mean_absolute_error(test_df["total_rides"], test_df["baseline_pred"])
print(f"Baseline Average Error (MAE): {round(baseline_mae):,} rides per day")

Baseline Average Error (MAE): 3,141 rides per day


#### Train Linear Regression 

In [15]:
df["date"] = pd.to_datetime(df["date"])

df["day_of_week"] = df["date"].dt.day_of_week

df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

features = [
    "temperature_2m_mean (°F)",
    "precipitation_sum (inch)",
    "snowfall_sum (inch)",
    "wind_speed_10m_max (mp/h)",
    "is_weekend"
]

X = df[features]
y = df["total_rides"]


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, shuffle=False
)

model = LinearRegression()
model.fit(X_train, y_train)
model_preds = model.predict(X_test)

# 5. Compare average daily error (MAE) against the baseline
model_mae = mean_absolute_error(y_test, model_preds)

print(f"Baseline Average Error (MAE): {round(baseline_mae):,} rides per day")
print(f"Linear Regression Error (MAE): {round(model_mae):,} rides per day")


Baseline Average Error (MAE): 3,141 rides per day
Linear Regression Error (MAE): 2,600 rides per day
